In [ ]:
import pandas as pd

df = pd.read_csv("Churn_Modelling.csv")

print(df.shape)
print(df.head())
df.info()
print(df.isnull().sum())

(10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4

In [ ]:
print(df.duplicated().sum())
print(df['Exited'].value_counts())

0
Exited
0    7963
1    2037
Name: count, dtype: int64


In [ ]:
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

for col in ['Geography', 'Gender', 'IsActiveMember', 'NumOfProducts', 'HasCrCard']:
    print(df.groupby(col)['Exited'].mean().round(3), '\n')


Geography
France     0.162
Germany    0.324
Spain      0.167
Name: Exited, dtype: float64 

Gender
Female    0.251
Male      0.165
Name: Exited, dtype: float64 

IsActiveMember
0    0.269
1    0.143
Name: Exited, dtype: float64 

NumOfProducts
1    0.277
2    0.076
3    0.827
4    1.000
Name: Exited, dtype: float64 

HasCrCard
0    0.208
1    0.202
Name: Exited, dtype: float64 



In [ ]:
print(df['NumOfProducts'].value_counts())
print(df['Geography'].value_counts())

NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64
Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64


In [ ]:
print(df['Geography'].value_counts())

print(df.groupby('NumOfProducts')['Exited'].agg(['count', 'sum', 'mean']).round(3))

print(df.groupby('Exited')[['Age', 'Balance', 'CreditScore', 'Tenure', 'EstimatedSalary']].mean().round(1))

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64
               count   sum   mean
NumOfProducts                    
1               5084  1409  0.277
2               4590   348  0.076
3                266   220  0.827
4                 60    60  1.000
         Age  Balance  CreditScore  Tenure  EstimatedSalary
Exited                                                     
0       37.4  72745.3        651.9     5.0          99738.4
1       44.8  91108.5        645.4     4.9         101465.7


In [ ]:
df['AgeGroup'] = pd.cut(df['Age'], bins=[17, 30, 40, 50, 60, 100],
                        labels=['18-30', '31-40', '41-50', '51-60', '60+'])
print(df.groupby('AgeGroup', observed=True)['Exited'].agg(['count', 'sum', 'mean']).round(3))

print((df['Balance'] == 0).mean())
print(df.groupby(df['Balance'] == 0)['Exited'].agg(['count', 'mean']).round(3))

          count  sum   mean
AgeGroup                   
18-30      1968  148  0.075
31-40      4451  538  0.121
41-50      2320  788  0.340
51-60       797  448  0.562
60+         464  115  0.248
0.3617
         count   mean
Balance              
False     6383  0.241
True      3617  0.138


In [ ]:
print(df.groupby(['AgeGroup', 'IsActiveMember'], observed=True)['Exited'].agg(['count', 'mean']).round(3))
print(df.groupby(['AgeGroup', 'Geography'], observed=True)['Exited'].agg(['count', 'mean']).round(3))
print(df.groupby(['Gender', 'Geography'])['Exited'].agg(['count', 'mean']).round(3))
print(df.groupby(['Gender', 'IsActiveMember'])['Exited'].agg(['count', 'mean']).round(3))


                         count   mean
AgeGroup IsActiveMember              
18-30    0                 962  0.101
         1                1006  0.051
31-40    0                2236  0.149
         1                2215  0.092
41-50    0                1226  0.416
         1                1094  0.254
51-60    0                 336  0.857
         1                 461  0.347
60+      0                  89  0.820
         1                 375  0.112
                    count   mean
AgeGroup Geography              
18-30    France      1038  0.049
         Germany      453  0.124
         Spain        477  0.086
31-40    France      2311  0.095
         Germany     1022  0.209
         Spain       1118  0.093
41-50    France      1067  0.281
         Germany      678  0.488
         Spain        575  0.273
51-60    France       367  0.526
         Germany      243  0.695
         Spain        187  0.460
60+      France       231  0.199
         Germany      113  0.389
         Spain  

In [ ]:
import sqlite3

conn = sqlite3.connect('churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)

query = "SELECT * FROM customers LIMIT 5;"
print(pd.read_sql(query, conn))

   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France  Female   42       2       0.00              1   
1          608     Spain  Female   41       1   83807.86              1   
2          502    France  Female   42       8  159660.80              3   
3          699    France  Female   39       1       0.00              2   
4          850     Spain  Female   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited AgeGroup  
0          1               1        101348.88       1    41-50  
1          0               1        112542.58       0    41-50  
2          1               0        113931.57       1    41-50  
3          0               0         93826.63       0    31-40  
4          1               1         79084.10       0    41-50  


In [ ]:
query = """
SELECT Geography, COUNT(*) as total_customers, SUM(Exited) as churned,
       ROUND(AVG(Exited), 3) as churn_rate
FROM customers
GROUP BY Geography
ORDER BY churn_rate DESC;
"""
print(pd.read_sql(query, conn))

  Geography  total_customers  churned  churn_rate
0   Germany             2509      814       0.324
1     Spain             2477      413       0.167
2    France             5014      810       0.162


In [ ]:
query = """
SELECT
    CASE
        WHEN Age < 30 THEN '<30'
        WHEN Age BETWEEN 30 AND 39 THEN '30-39'
        WHEN Age BETWEEN 40 AND 49 THEN '40-49'
        WHEN Age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS age_band,
    COUNT(*) AS total_customers,
    SUM(Exited) AS churned,
    ROUND(AVG(Exited), 3) AS churn_rate
FROM customers
GROUP BY age_band
ORDER BY churn_rate DESC;
"""
print(pd.read_sql(query, conn))

  age_band  total_customers  churned  churn_rate
0    50-59              869      487       0.560
1    40-49             2618      806       0.308
2      60+              526      147       0.279
3    30-39             4346      473       0.109
4      <30             1641      124       0.076


In [ ]:
query = """
SELECT Geography,
       ROUND(AVG(Balance), 0) AS avg_balance,
       ROUND(AVG(NumOfProducts), 2) AS avg_products,
       ROUND(AVG(IsActiveMember), 3) AS active_rate,
       ROUND(AVG(Age), 1) AS avg_age,
       ROUND(AVG(Exited), 3) AS churn_rate
FROM customers
GROUP BY Geography;
"""
print(pd.read_sql(query, conn))

  Geography  avg_balance  avg_products  active_rate  avg_age  churn_rate
0    France      62093.0          1.53        0.517     38.5       0.162
1   Germany     119730.0          1.52        0.497     39.8       0.324
2     Spain      61818.0          1.54        0.530     38.9       0.167


In [ ]:
query = """
SELECT Geography,
       CASE WHEN Balance = 0 THEN 'Zero balance' ELSE 'Has balance' END AS balance_group,
       COUNT(*) AS total_customers,
       ROUND(AVG(Exited), 3) AS churn_rate
FROM customers
GROUP BY Geography, balance_group
ORDER BY Geography, balance_group;
"""
print(pd.read_sql(query, conn))

  Geography balance_group  total_customers  churn_rate
0    France   Has balance             2596       0.182
1    France  Zero balance             2418       0.139
2   Germany   Has balance             2509       0.324
3     Spain   Has balance             1278       0.196
4     Spain  Zero balance             1199       0.136


In [ ]:
query = """
SELECT Geography,
       CASE WHEN Age < 40 THEN 'Under 40' ELSE '40+' END AS age_group,
       COUNT(*) AS total_customers,
       ROUND(AVG(Exited), 3) AS churn_rate
FROM customers
GROUP BY Geography, age_group
ORDER BY Geography, age_group;
"""
print(pd.read_sql(query, conn))

  Geography age_group  total_customers  churn_rate
0    France       40+             1898       0.307
1    France  Under 40             3116       0.073
2   Germany       40+             1140       0.508
3   Germany  Under 40             1369       0.172
4     Spain       40+              975       0.286
5     Spain  Under 40             1502       0.089


In [ ]:
query = """
SELECT NumOfProducts,
       IsActiveMember,
       COUNT(*) AS total_customers,
       SUM(Exited) AS churned,
       ROUND(AVG(Exited), 3) AS churn_rate
FROM customers
GROUP BY NumOfProducts, IsActiveMember
ORDER BY NumOfProducts, IsActiveMember;
"""
print(pd.read_sql(query, conn))

   NumOfProducts  IsActiveMember  total_customers  churned  churn_rate
0              1               0             2521      924       0.367
1              1               1             2563      485       0.189
2              2               0             2144      212       0.099
3              2               1             2446      136       0.056
4              3               0              153      135       0.882
5              3               1              113       85       0.752
6              4               0               31       31       1.000
7              4               1               29       29       1.000


In [ ]:
query = """
WITH seg AS (
    SELECT NumOfProducts,
           IsActiveMember,
           COUNT(*) AS total_customers,
           SUM(Exited) AS churned
    FROM customers
    GROUP BY NumOfProducts, IsActiveMember
)
SELECT NumOfProducts,
       IsActiveMember,
       total_customers,
       churned,
       ROUND(100.0 * churned / total_customers, 1) AS churn_rate_pct,
       ROUND(100.0 * churned / SUM(churned) OVER (), 1) AS pct_of_all_churn,
       RANK() OVER (ORDER BY churned DESC) AS volume_rank
FROM seg
ORDER BY volume_rank;
"""
print(pd.read_sql(query, conn))

   NumOfProducts  IsActiveMember  total_customers  churned  churn_rate_pct  \
0              1               0             2521      924            36.7   
1              1               1             2563      485            18.9   
2              2               0             2144      212             9.9   
3              2               1             2446      136             5.6   
4              3               0              153      135            88.2   
5              3               1              113       85            75.2   
6              4               0               31       31           100.0   
7              4               1               29       29           100.0   

   pct_of_all_churn  volume_rank  
0              45.4            1  
1              23.8            2  
2              10.4            3  
3               6.7            4  
4               6.6            5  
5               4.2            6  
6               1.5            7  
7               1